In [12]:
import os
import io
import random
import numpy as np
import matplotlib.pyplot as plt
import pyarrow as pa
import pyarrow.parquet as pq
from imblearn.over_sampling import (
    SMOTE,
    SMOTENC,
    SMOTEN,
    SVMSMOTE,
    KMeansSMOTE,
    BorderlineSMOTE,
    RandomOverSampler,
    ADASYN,
)

from functools import reduce
from concurrent.futures import ThreadPoolExecutor

In [13]:
DATA_DIR = "../include/data"

In [14]:
# # cloud
# URL = "abfss://{FOLDER_NAME}@sgppipelinesa.dfs.core.windows.net"
# SILVER_FOLDER_NAME = "sgppipelinesa-silver"
# SUB_FOLDER_NAME = "stage-02"
# SILVER_DATA_DIR = os.path.join(URL, "{SUB_FOLDER_NAME}").replace("\\", "/")
# SILVER_DATA_DIR

# local
SILVER_FOLDER_NAME = "silver"
SUB_FOLDER_NAME = "stage-03"
SILVER_DATA_DIR = os.path.join("{DATA_DIR}", "{FOLDER_NAME}", "{SUB_FOLDER_NAME}").replace("\\", "/")
SILVER_DATA_DIR

'{DATA_DIR}/{FOLDER_NAME}/{SUB_FOLDER_NAME}'

# Run following cells if you want to run signal feature augmentation in a distributed manner using apache spark

In [15]:
import pyspark
import pyspark.sql.functions as F

from pyspark.sql import SparkSession, Window, Row, DataFrame
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, BucketedRandomProjectionLSH, VectorSlicer, StringIndexer, Imputer
from pyspark.ml.linalg import Vectors, VectorUDT, SparseVector, DenseVector

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [329]:
spark = SparkSession.builder.appName("app")\
    .config("spark.driver.memory", "16g")\
    .config("spark.executor.memory", "4g")\
    .config("spark.executor.cores", "2")\
    .config("spark.executor.instances", "3")\
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "100")\
    .getOrCreate()

# spark = SparkSession.builder.appName("app")\
#     .getOrCreate()

In [330]:
train_data_df = spark.read.format("parquet").load(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "train_data.parquet"
    ).replace("\\", "/")
)

In [331]:
train_data_df.cache()

DataFrame[subjectId: string, freq_kurt_imp: double, freq_skew_imp: double, freq_entropy_imp: double, freq_mean_imp: double, freq_median_imp: double, freq_mode_imp: double, freq_min_imp: double, freq_max_imp: double, freq_var_imp: double, freq_stddev_imp: double, freq_first_quart_imp: double, freq_third_quart_imp: double, freq_range_imp: double, freq_inter_quart_range_imp: double, zcr_imp: double, poly_feat_1_imp: double, poly_feat_2_imp: double, spec_cent_imp: double, spec_bw_imp: double, spec_flat_imp: double, spec_roll_imp: double, mel_spec_mean_imp: double, mel_spec_median_imp: double, mel_spec_mode_imp: double, mel_spec_mode_cnt_imp: double, mel_spec_min_imp: double, mel_spec_max_imp: double, mel_spec_range_imp: double, mel_spec_var_imp: double, mel_spec_std_imp: double, mel_spec_first_quart_imp: double, mel_spec_third_quart_imp: double, mel_spec_inter_quart_range_imp: double, mel_spec_entropy_imp: double, mel_spec_kurt_imp: double, mel_spec_skew_imp: double, mel_spec_db_mean_imp: 

In [332]:
train_data_df.show()

+--------------------+------------------+------------------+-----------------+--------------------+-------------------+-----------------+------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-------------------+--------------------+---------------------+--------------------+------------------+------------------+----------------+------------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------

In [333]:
val_data_df = spark.read.format("parquet").load(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "val_data.parquet"
    ).replace("\\", "/")
)

In [334]:
val_data_df.cache()

DataFrame[subjectId: string, freq_kurt_imp: double, freq_skew_imp: double, freq_entropy_imp: double, freq_mean_imp: double, freq_median_imp: double, freq_mode_imp: double, freq_min_imp: double, freq_max_imp: double, freq_var_imp: double, freq_stddev_imp: double, freq_first_quart_imp: double, freq_third_quart_imp: double, freq_range_imp: double, freq_inter_quart_range_imp: double, zcr_imp: double, poly_feat_1_imp: double, poly_feat_2_imp: double, spec_cent_imp: double, spec_bw_imp: double, spec_flat_imp: double, spec_roll_imp: double, mel_spec_mean_imp: double, mel_spec_median_imp: double, mel_spec_mode_imp: double, mel_spec_mode_cnt_imp: double, mel_spec_min_imp: double, mel_spec_max_imp: double, mel_spec_range_imp: double, mel_spec_var_imp: double, mel_spec_std_imp: double, mel_spec_first_quart_imp: double, mel_spec_third_quart_imp: double, mel_spec_inter_quart_range_imp: double, mel_spec_entropy_imp: double, mel_spec_kurt_imp: double, mel_spec_skew_imp: double, mel_spec_db_mean_imp: 

In [335]:
val_data_df.show()

+--------------------+-----------------+------------------+-----------------+--------------------+------------------+-----------------+------------------+-----------------+--------------------+-------------------+--------------------+--------------------+-----------------+--------------------------+-------------------+--------------------+------------------+------------------+------------------+--------------------+-----------------+------------------+-------------------+--------------------+---------------------+--------------------+-----------------+------------------+----------------+------------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------

In [336]:
test_data_df = spark.read.format("parquet").load(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "test_data.parquet"
    ).replace("\\", "/")
)

In [337]:
test_data_df.cache()

DataFrame[subjectId: string, freq_kurt_imp: double, freq_skew_imp: double, freq_entropy_imp: double, freq_mean_imp: double, freq_median_imp: double, freq_mode_imp: double, freq_min_imp: double, freq_max_imp: double, freq_var_imp: double, freq_stddev_imp: double, freq_first_quart_imp: double, freq_third_quart_imp: double, freq_range_imp: double, freq_inter_quart_range_imp: double, zcr_imp: double, poly_feat_1_imp: double, poly_feat_2_imp: double, spec_cent_imp: double, spec_bw_imp: double, spec_flat_imp: double, spec_roll_imp: double, mel_spec_mean_imp: double, mel_spec_median_imp: double, mel_spec_mode_imp: double, mel_spec_mode_cnt_imp: double, mel_spec_min_imp: double, mel_spec_max_imp: double, mel_spec_range_imp: double, mel_spec_var_imp: double, mel_spec_std_imp: double, mel_spec_first_quart_imp: double, mel_spec_third_quart_imp: double, mel_spec_inter_quart_range_imp: double, mel_spec_entropy_imp: double, mel_spec_kurt_imp: double, mel_spec_skew_imp: double, mel_spec_db_mean_imp: 

In [338]:
test_data_df.show()

+--------------------+--------------------+-------------------+------------------+--------------------+-------------------+------------------+----------------+-----------------+--------------------+-------------------+--------------------+--------------------+-----------------+--------------------------+-------------------+--------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-------------------+--------------------+---------------------+--------------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+------------------------

# remove subjectId column from data frames

In [339]:
train_data_df = train_data_df.drop(*["subjectId"])

In [340]:
val_data_df = val_data_df.drop(*["subjectId"])

In [341]:
test_data_df = test_data_df.drop(*["subjectId"])

# convert the label categorical columns to a numerical value, 0 for male and 1 for female 

In [342]:
label_cond = F.when(
    F.col("label") == "male",
    0
).otherwise(1)

In [343]:
train_data_df = train_data_df.withColumn("label", label_cond)
train_data_df.show()

+------------------+------------------+-----------------+--------------------+-------------------+-----------------+------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-------------------+--------------------+---------------------+--------------------+------------------+------------------+----------------+------------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+--------------

In [344]:
val_data_df = val_data_df.withColumn("label", label_cond)
val_data_df.show()

+-----------------+------------------+-----------------+--------------------+------------------+-----------------+------------------+-----------------+--------------------+-------------------+--------------------+--------------------+-----------------+--------------------------+-------------------+--------------------+------------------+------------------+------------------+--------------------+-----------------+------------------+-------------------+--------------------+---------------------+--------------------+-----------------+------------------+----------------+------------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+--------------------

In [345]:
test_data_df = test_data_df.withColumn("label", label_cond)
test_data_df.show()

+--------------------+-------------------+------------------+--------------------+-------------------+------------------+----------------+-----------------+--------------------+-------------------+--------------------+--------------------+-----------------+--------------------------+-------------------+--------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-------------------+--------------------+---------------------+--------------------+----------------+------------------+----------------+----------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+-----------------

In [346]:
train_data_df.count(), val_data_df.count(), test_data_df.count()

(213111, 47500, 54036)

# how string index works

In [347]:
toy_df = train_data_df.sample(withReplacement=False, fraction=0.5)
toy_df.show()

+------------------+--------------------+------------------+--------------------+-------------------+------------------+------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-------------------+--------------------+---------------------+--------------------+-----------------+------------------+----------------+------------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+-----------

In [348]:
toy_df = toy_df.withColumn("some_cat_col", F.lit("some literal"))
toy_df.show()

+------------------+--------------------+------------------+--------------------+-------------------+------------------+------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-------------------+--------------------+---------------------+--------------------+-----------------+------------------+----------------+------------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+-----------

In [349]:
num_cols = list(filter(lambda col: not "label" in col, train_data_df.columns))
num_cols

['freq_kurt_imp',
 'freq_skew_imp',
 'freq_entropy_imp',
 'freq_mean_imp',
 'freq_median_imp',
 'freq_mode_imp',
 'freq_min_imp',
 'freq_max_imp',
 'freq_var_imp',
 'freq_stddev_imp',
 'freq_first_quart_imp',
 'freq_third_quart_imp',
 'freq_range_imp',
 'freq_inter_quart_range_imp',
 'zcr_imp',
 'poly_feat_1_imp',
 'poly_feat_2_imp',
 'spec_cent_imp',
 'spec_bw_imp',
 'spec_flat_imp',
 'spec_roll_imp',
 'mel_spec_mean_imp',
 'mel_spec_median_imp',
 'mel_spec_mode_imp',
 'mel_spec_mode_cnt_imp',
 'mel_spec_min_imp',
 'mel_spec_max_imp',
 'mel_spec_range_imp',
 'mel_spec_var_imp',
 'mel_spec_std_imp',
 'mel_spec_first_quart_imp',
 'mel_spec_third_quart_imp',
 'mel_spec_inter_quart_range_imp',
 'mel_spec_entropy_imp',
 'mel_spec_kurt_imp',
 'mel_spec_skew_imp',
 'mel_spec_db_mean_imp',
 'mel_spec_db_median_imp',
 'mel_spec_db_mode_imp',
 'mel_spec_db_mode_cnt_imp',
 'mel_spec_db_min_imp',
 'mel_spec_db_max_imp',
 'mel_spec_db_range_imp',
 'mel_spec_db_var_imp',
 'mel_spec_db_std_imp',
 'm

In [350]:
assembler = VectorAssembler(inputCols=num_cols, outputCol="features")
# assembler.setHandleInvalid("keep")

In [351]:
cat_cols = ["some_cat_col"]

In [352]:
target_col = "label"

In [353]:
list(set(cat_cols) - set(["label"]))

['some_cat_col']

In [354]:
# index the string cols, except possibly for the label col
index_suffix = "_index"
cat_cols_to_vectorize = list(set(cat_cols) - set([target_col]))
assemble_stages = [
    StringIndexer(inputCol=column, outputCol=column + index_suffix).fit(toy_df) 
    for column in cat_cols
]

In [355]:
# add the stage of numerical vector assembler
assemble_stages.append(assembler)

#### note that string indexers, scalers, assemblers rely heavily on its columsn not being null so make sure all null values are imputed first in order to avoid `SparkException: Values to assemble cannot be null`  

In [356]:
assemble_stages

[StringIndexerModel: uid=StringIndexer_c0f3d334eddc, handleInvalid=error,
 VectorAssembler_7e8ed621c87c]

In [357]:

pipeline = Pipeline(stages=assemble_stages)
model = pipeline.fit(train_data_df)

In [358]:
toy_df_vec = model.transform(toy_df)

In [359]:
toy_df_vec.show()

+------------------+--------------------+------------------+--------------------+-------------------+------------------+------------------+-----------------+--------------------+--------------------+--------------------+--------------------+-----------------+--------------------------+--------------------+--------------------+------------------+------------------+------------------+--------------------+------------------+------------------+-------------------+--------------------+---------------------+--------------------+-----------------+------------------+----------------+------------------+------------------------+------------------------+------------------------------+--------------------+------------------+------------------+--------------------+----------------------+--------------------+------------------------+-------------------+-------------------+---------------------+-------------------+-------------------+---------------------------+---------------------------+-----------

In [360]:
# drop original num cols and cat cols
drop_cols = num_cols + cat_cols
keep_cols = [a for a in toy_df_vec.columns if a not in drop_cols]
keep_cols

['label', 'some_cat_col_index', 'features']

In [361]:
vectorized = toy_df_vec.select(*keep_cols) \
.withColumn('label', toy_df_vec[target_col])
vectorized.show()

+-----+------------------+--------------------+
|label|some_cat_col_index|            features|
+-----+------------------+--------------------+
|    0|               0.0|[20.5705867663724...|
|    0|               0.0|[20.1761696819793...|
|    0|               0.0|[19.8838103731754...|
|    0|               0.0|[22.2407267785526...|
|    0|               0.0|[24.2508943038969...|
|    0|               0.0|[24.1674986900393...|
|    0|               0.0|[23.9823953831754...|
|    0|               0.0|[23.1666870845078...|
|    0|               0.0|[22.7826405522046...|
|    0|               0.0|[21.9946418160058...|
|    0|               0.0|[21.2664837820400...|
|    0|               0.0|[19.5875564348823...|
|    0|               0.0|[19.1185366217878...|
|    0|               0.0|[19.2288179920592...|
|    0|               0.0|[19.4662000764536...|
|    0|               0.0|[19.7949994074827...|
|    0|               0.0|[19.0587329648620...|
|    0|               0.0|[18.0190290234

In [362]:
vectorized.count()

106498

In [363]:
vectorized.cache()

DataFrame[label: int, some_cat_col_index: double, features: vector]

In [364]:
min_class = vectorized.where(F.col("label") == 1)
maj_class = vectorized.where(F.col("label") == 0)
min_class

DataFrame[label: int, some_cat_col_index: double, features: vector]

In [365]:
min_class.show()

+-----+------------------+--------------------+
|label|some_cat_col_index|            features|
+-----+------------------+--------------------+
|    1|               0.0|[6.45531921352894...|
|    1|               0.0|[5.55886122283727...|
|    1|               0.0|[6.08798921699230...|
|    1|               0.0|[5.62201682265669...|
|    1|               0.0|[5.27358033915110...|
|    1|               0.0|[4.90714335319906...|
|    1|               0.0|[4.11207298583265...|
|    1|               0.0|[4.97323468214916...|
|    1|               0.0|[8.6269840244153,...|
|    1|               0.0|[8.43634474063953...|
|    1|               0.0|[7.52805263657426...|
|    1|               0.0|[7.01040389849684...|
|    1|               0.0|[7.67596275988288...|
|    1|               0.0|[5.84935032335276...|
|    1|               0.0|[-1.0246216397434...|
|    1|               0.0|[5.27240064330778...|
|    1|               0.0|[2.39702416395806...|
|    1|               0.0|[2.50981413962

In [ ]:
def subtract_vector_fn(arr):
    a = arr[0]
    b = arr[1]
    
    if isinstance(a, SparseVector):
        a = a.toArray()
        
    if isinstance(b, SparseVector):
        b = b.toArray()
    
    return DenseVector(random.uniform(0, 1)*(a-b))
    
def add_vector_fn(arr):
    a = arr[0]
    b = arr[1]
    
    if isinstance(a, SparseVector):
        a = a.toArray()
        
    if isinstance(b, SparseVector):
        b = b.toArray()
    
    return DenseVector(a+b)

def smote(vectorized_sdf, smote_config):
    '''
    contains logic to perform smote oversampling, given a spark df with 2 classes
    inputs:
    * vectorized_sdf: cat cols are already stringindexed, num cols are assembled into 'features' vector
      df target col should be 'label'
    * smote_config: config obj containing smote parameters
    output:
    * oversampled_df: spark df after smote oversampling
    '''
    dataInput_min = vectorized_sdf[vectorized_sdf['label'] == smote_config.positive_label]
    dataInput_maj = vectorized_sdf[vectorized_sdf['label'] == smote_config.negative_label]
    
    # LSH, bucketed random projection
    brp = BucketedRandomProjectionLSH(
        inputCol="features", 
        outputCol="hashes",
        seed=int(smote_config.seed),
        bucketLength=float(smote_config.bucketLength)
    )

    # smote only applies on existing minority instances    
    model = brp.fit(dataInput_min)
    model.transform(dataInput_min)

    # here distance is calculated from brp's param inputCol
    self_join_w_distance = model.approxSimilarityJoin(dataInput_min, dataInput_min, float('inf'), distCol="EuclideanDistance")

    # remove self-comparison (distance 0)
    self_join_w_distance = self_join_w_distance.filter(self_join_w_distance.EuclideanDistance > 0)

    over_original_rows = Window.partitionBy("datasetA").orderBy("EuclideanDistance")

    self_similarity_df = self_join_w_distance.withColumn("r_num", F.row_number().over(over_original_rows))

    self_similarity_df_selected = self_similarity_df.filter(self_similarity_df.r_num <= int(smote_config.k))

    over_original_rows_no_order = Window.partitionBy('datasetA')

    # list to store batches of synthetic data
    res = []
    
    # two udf for vector add and subtract, subtraction include a random factor [0,1]
    subtract_vector_udf = F.udf(subtract_vector_fn, VectorUDT())
    add_vector_udf = F.udf(add_vector_fn, VectorUDT())
    
    # retain original columns
    original_cols = dataInput_min.columns
    
    for i in range(int(smote_config.multiplier)):
        print("generating batch %s of synthetic instances"%i)
        # logic to randomly select neighbour: pick the largest random number generated row as the neighbour
        df_random_sel = self_similarity_df_selected\
                            .withColumn("rand", F.rand())\
                            .withColumn('max_rand', F.max('rand').over(over_original_rows_no_order))\
                            .where(F.col('rand') == F.col('max_rand')).drop(*['max_rand','rand','r_num'])
        # create synthetic feature numerical part
        df_vec_diff = df_random_sel\
            .select('*', subtract_vector_udf(F.array('datasetA.features', 'datasetB.features')).alias('vec_diff'))
        df_vec_modified = df_vec_diff\
            .select('*', add_vector_udf(F.array('datasetB.features', 'vec_diff')).alias('features'))
        
        # for categorical cols, either pick original or the neighbour's cat values
        for c in original_cols:
            # randomly select neighbour or original data
            col_sub = random.choice(['datasetA','datasetB'])
            val = "{0}.{1}".format(col_sub,c)
            if c != 'features':
                # do not unpack original numerical features
                df_vec_modified = df_vec_modified.withColumn(c,F.col(val))
        
        # this df_vec_modified is the synthetic minority instances,
        df_vec_modified = df_vec_modified.drop(*['datasetA', 'datasetB', 'vec_diff', 'EuclideanDistance'])
        
        res.append(df_vec_modified)
    
    dfunion = reduce(DataFrame.union, res)
    dfunion = dfunion.union(dataInput_min.select(dfunion.columns))\
        .sort(F.rand(seed=smote_config.seed))\
        .withColumn('row_number', F.row_number().over(Window.orderBy(F.lit('A'))))
    
    dataInput_maj = dataInput_maj.withColumn('row_number', F.row_number().over(Window.orderBy(F.lit('A'))))
    
    # union synthetic instances with original full (both minority and majority) df
    oversampled_df = dfunion.union(dataInput_maj.select(dfunion.columns))
    
    return oversampled_df.sort('row_number').drop(*['row_number'])

class SmoteConfig:
    def __init__(self, seed, bucketLength, k, multiplier, positive_label, negative_label):
        """"
        The bucket length is a parameter that determines the step 
        size for the number of synthetic samples to generate during 
        SMOTE. Basically, it controls the granularity of oversampling. 
        In short, the bucket length controls the spacing between 
        synthetic samples in terms of their proximity to the original 
        minority class instances.

        The multiplier is to determine how many synthetic samples to 
        create. It controls the total number of samples to oversample.

        “k” refers to the number of nearest neighbors used to select 
        the neighboring instances when generating synthetic samples 
        for the minority class. It plays a crucial role in determining 
        the characteristics of the synthetic samples. Smaller k will 
        have less diversity. Higher k will have more diversity.
        """
        self.seed = seed
        self.bucketLength = bucketLength
        self.k = k
        self.multiplier = multiplier
        self.positive_label = positive_label
        self.negative_label = negative_label

In [368]:
smote_config = SmoteConfig(
    seed=76, 
    bucketLength=200, 
    k=10, 
    multiplier=25, 
    positive_label=1, 
    negative_label=0)

In [375]:
test_df_augmented = smote(vectorized, smote_config)

generating batch 0 of synthetic instances
generating batch 1 of synthetic instances
generating batch 2 of synthetic instances
generating batch 3 of synthetic instances
generating batch 4 of synthetic instances
generating batch 5 of synthetic instances
generating batch 6 of synthetic instances
generating batch 7 of synthetic instances
generating batch 8 of synthetic instances
generating batch 9 of synthetic instances
generating batch 10 of synthetic instances
generating batch 11 of synthetic instances
generating batch 12 of synthetic instances
generating batch 13 of synthetic instances
generating batch 14 of synthetic instances
generating batch 15 of synthetic instances
generating batch 16 of synthetic instances
generating batch 17 of synthetic instances
generating batch 18 of synthetic instances
generating batch 19 of synthetic instances
generating batch 20 of synthetic instances
generating batch 21 of synthetic instances
generating batch 22 of synthetic instances
generating batch 23 o

In [ ]:
test_df_augmented.count()

# Run if the following if spark cluster is not available to augment signal features in a distributed manner

In [ ]:
train_data_df = pq.read_table(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "train_data.parquet"
    ).replace("\\", "/")
)
train_data_df

pyarrow.Table
freq_kurt_imp: double
freq_skew_imp: double
freq_entropy_imp: double
freq_mean_imp: double
freq_median_imp: double
freq_mode_imp: double
freq_min_imp: double
freq_max_imp: double
freq_var_imp: double
freq_stddev_imp: double
freq_first_quart_imp: double
freq_third_quart_imp: double
freq_range_imp: double
freq_inter_quart_range_imp: double
zcr_imp: double
poly_feat_1_imp: double
poly_feat_2_imp: double
spec_cent_imp: double
spec_bw_imp: double
spec_flat_imp: double
spec_roll_imp: double
mel_spec_mean_imp: double
mel_spec_median_imp: double
mel_spec_mode_imp: double
mel_spec_mode_cnt_imp: double
mel_spec_min_imp: double
mel_spec_max_imp: double
mel_spec_range_imp: double
mel_spec_var_imp: double
mel_spec_std_imp: double
mel_spec_first_quart_imp: double
mel_spec_third_quart_imp: double
mel_spec_inter_quart_range_imp: double
mel_spec_entropy_imp: double
mel_spec_kurt_imp: double
mel_spec_skew_imp: double
mel_spec_db_mean_imp: double
mel_spec_db_median_imp: double
mel_spec_db_m

In [20]:
val_data_df = pq.read_table(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "val_data.parquet"
    ).replace("\\", "/")
)
val_data_df

pyarrow.Table
freq_kurt_imp: double
freq_skew_imp: double
freq_entropy_imp: double
freq_mean_imp: double
freq_median_imp: double
freq_mode_imp: double
freq_min_imp: double
freq_max_imp: double
freq_var_imp: double
freq_stddev_imp: double
freq_first_quart_imp: double
freq_third_quart_imp: double
freq_range_imp: double
freq_inter_quart_range_imp: double
zcr_imp: double
poly_feat_1_imp: double
poly_feat_2_imp: double
spec_cent_imp: double
spec_bw_imp: double
spec_flat_imp: double
spec_roll_imp: double
mel_spec_mean_imp: double
mel_spec_median_imp: double
mel_spec_mode_imp: double
mel_spec_mode_cnt_imp: double
mel_spec_min_imp: double
mel_spec_max_imp: double
mel_spec_range_imp: double
mel_spec_var_imp: double
mel_spec_std_imp: double
mel_spec_first_quart_imp: double
mel_spec_third_quart_imp: double
mel_spec_inter_quart_range_imp: double
mel_spec_entropy_imp: double
mel_spec_kurt_imp: double
mel_spec_skew_imp: double
mel_spec_db_mean_imp: double
mel_spec_db_median_imp: double
mel_spec_db_m

In [21]:
test_data_df = pq.read_table(
    os.path.join(
        SILVER_DATA_DIR.format(
            DATA_DIR=DATA_DIR,
            FOLDER_NAME=SILVER_FOLDER_NAME,
            SUB_FOLDER_NAME=SUB_FOLDER_NAME
        ),
        "test_data.parquet"
    ).replace("\\", "/")
)
test_data_df

pyarrow.Table
freq_kurt_imp: double
freq_skew_imp: double
freq_entropy_imp: double
freq_mean_imp: double
freq_median_imp: double
freq_mode_imp: double
freq_min_imp: double
freq_max_imp: double
freq_var_imp: double
freq_stddev_imp: double
freq_first_quart_imp: double
freq_third_quart_imp: double
freq_range_imp: double
freq_inter_quart_range_imp: double
zcr_imp: double
poly_feat_1_imp: double
poly_feat_2_imp: double
spec_cent_imp: double
spec_bw_imp: double
spec_flat_imp: double
spec_roll_imp: double
mel_spec_mean_imp: double
mel_spec_median_imp: double
mel_spec_mode_imp: double
mel_spec_mode_cnt_imp: double
mel_spec_min_imp: double
mel_spec_max_imp: double
mel_spec_range_imp: double
mel_spec_var_imp: double
mel_spec_std_imp: double
mel_spec_first_quart_imp: double
mel_spec_third_quart_imp: double
mel_spec_inter_quart_range_imp: double
mel_spec_entropy_imp: double
mel_spec_kurt_imp: double
mel_spec_skew_imp: double
mel_spec_db_mean_imp: double
mel_spec_db_median_imp: double
mel_spec_db_m

In [24]:
feat_cols = list(filter(lambda feat_col: not "label" in feat_col, train_data_df.column_names))
feat_cols

['freq_kurt_imp',
 'freq_skew_imp',
 'freq_entropy_imp',
 'freq_mean_imp',
 'freq_median_imp',
 'freq_mode_imp',
 'freq_min_imp',
 'freq_max_imp',
 'freq_var_imp',
 'freq_stddev_imp',
 'freq_first_quart_imp',
 'freq_third_quart_imp',
 'freq_range_imp',
 'freq_inter_quart_range_imp',
 'zcr_imp',
 'poly_feat_1_imp',
 'poly_feat_2_imp',
 'spec_cent_imp',
 'spec_bw_imp',
 'spec_flat_imp',
 'spec_roll_imp',
 'mel_spec_mean_imp',
 'mel_spec_median_imp',
 'mel_spec_mode_imp',
 'mel_spec_mode_cnt_imp',
 'mel_spec_min_imp',
 'mel_spec_max_imp',
 'mel_spec_range_imp',
 'mel_spec_var_imp',
 'mel_spec_std_imp',
 'mel_spec_first_quart_imp',
 'mel_spec_third_quart_imp',
 'mel_spec_inter_quart_range_imp',
 'mel_spec_entropy_imp',
 'mel_spec_kurt_imp',
 'mel_spec_skew_imp',
 'mel_spec_db_mean_imp',
 'mel_spec_db_median_imp',
 'mel_spec_db_mode_imp',
 'mel_spec_db_mode_cnt_imp',
 'mel_spec_db_min_imp',
 'mel_spec_db_max_imp',
 'mel_spec_db_range_imp',
 'mel_spec_db_var_imp',
 'mel_spec_db_std_imp',
 'm

In [26]:
# oversampling the train dataset using SMOTE
smt = SMOTE()

In [38]:
train_data_df.select(["label"])

pyarrow.Table
label: int32
----
label: [[0,0,0,0,0,...,0,0,0,0,0],[0,0,0,0,0,...,0,0,0,0,0]]

In [43]:
train_data_df.select(feat_cols)

pyarrow.Table
freq_kurt_imp: double
freq_skew_imp: double
freq_entropy_imp: double
freq_mean_imp: double
freq_median_imp: double
freq_mode_imp: double
freq_min_imp: double
freq_max_imp: double
freq_var_imp: double
freq_stddev_imp: double
freq_first_quart_imp: double
freq_third_quart_imp: double
freq_range_imp: double
freq_inter_quart_range_imp: double
zcr_imp: double
poly_feat_1_imp: double
poly_feat_2_imp: double
spec_cent_imp: double
spec_bw_imp: double
spec_flat_imp: double
spec_roll_imp: double
mel_spec_mean_imp: double
mel_spec_median_imp: double
mel_spec_mode_imp: double
mel_spec_mode_cnt_imp: double
mel_spec_min_imp: double
mel_spec_max_imp: double
mel_spec_range_imp: double
mel_spec_var_imp: double
mel_spec_std_imp: double
mel_spec_first_quart_imp: double
mel_spec_third_quart_imp: double
mel_spec_inter_quart_range_imp: double
mel_spec_entropy_imp: double
mel_spec_kurt_imp: double
mel_spec_skew_imp: double
mel_spec_db_mean_imp: double
mel_spec_db_median_imp: double
mel_spec_db_m

In [ ]:
train_input_sm, train_output_sm = smt.fit_resample(train_data_df.select(feat_cols).to_numpy(), train_data_df.select("label"))
val_input_sm, val_output_sm = smt.fit_resample(val_data_df.select(feat_cols).to_numpy(), val_data_df.select("label"))
test_input_sm, test_output_sm = smt.fit_resample(test_data_df.select(feat_cols).to_numpy(), test_data_df.select("label"))

AttributeError: 'pyarrow.lib.Table' object has no attribute 'to_numpy'